In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath("/Users/charles/Documents/PhD/Analysis/ieeg-pipeline")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from analysis.decoding.config import *
from analysis.decoding.core_classes.loader import GroupLoader
from analysis.decoding.decoding_analysis import DecodingAnalysis
from analysis.decoding.core_classes.process_features import Features
from analysis.decoding.geometry import Geometry
from tqdm import tqdm

plt.style.use('seaborn-v0_8-poster') 

subjects = [2, 3, 4, 5, 8, 9, 12, 14, 16, 19, 20, 23, 25, 28]
group_selection = GroupLoader(subjects)
group_selection.load_anatomy()
event_name = "fb"
tmin = -1.5
tmax = 1.5
complete_beh = pd.DataFrame()
complete_tfr = {subject: None for subject in subjects}
complete_baseline = {subject: None for subject in subjects}
# for subject in subjects:
for i, subject in enumerate(subjects):
# subject = 28
    power_path = os.path.join(DATA_DIR, f"sub-{int(subject):03}", "preprocessed", "aligned", f"sub-{int(subject):03}_tfr-realign-{event_name}_{tmin}-{tmax}_power.npy")
    metadata_path = os.path.join(os.path.join(DATA_DIR, f"sub-{int(subject):03}", "preprocessed", "aligned", f"sub-{int(subject):03}_tfr-realign-{event_name}-{tmin}-{tmax}_metadata.json"))
    with open(metadata_path, 'r', encoding='utf-8') as f: 
        metadata = json.load(f)
    n_epochs = metadata["n_epochs"]
    n_channels = metadata["n_channels"]
    event_to_keep = metadata["keep_events"]
    complete_tfr[subject] = np.load(power_path, mmap_mode='r')
    #np.load(power_path, mmap_mode='r')#np.memmap(power_path, mode='r', shape=(n_epochs, n_channels, n_freqs, int((tmax - tmin)*sr_decimated)), dtype=np.float16)
    baseline_path = os.path.join(DATA_DIR, f"sub-{int(subject):03}", "preprocessed", "timefreq", f"sub-{int(subject):03}_tfr-baseline.npy")
    baseline = np.load(baseline_path)
    complete_baseline[subject] = baseline[event_to_keep]
    beh_path = os.path.join(os.path.join(DATA_DIR, f"sub-{int(subject):03}", "preprocessed", "aligned", f"sub-{int(subject):03}_task-stratinf_beh-aligned.csv"))
    beh = pd.read_csv(beh_path)
    beh["index"] = beh.index
    complete_beh = pd.concat([complete_beh, beh], ignore_index=True)



[fetch_surf_fsaverage] Dataset found in /Users/charles/nilearn_data/fsaverage
[fetch_surf_fsaverage] Dataset found in /Users/charles/nilearn_data/fsaverage


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.impute import SimpleImputer



In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
clf = LinearSVC(max_iter=10000)
pipeline = make_pipeline(StandardScaler(), clf)
imputer = SimpleImputer(strategy="mean")  # or "median"

In [ ]:

complete_tfr_band = {subject: None for subject in subjects}

In [ ]:
complete_baseline_band = {subject: None for subject in subjects}
# complete_baseline

In [ ]:
for subject in subjects:
    print(f"Subject {subject}")
    tmp_tfr = complete_tfr[subject]
    tmp_baseline = complete_baseline[subject]
    n_epochs = tmp_tfr.shape[0]
    n_channels = tmp_tfr.shape[1]
    n_time = tmp_tfr.shape[-1]
    power_bands = np.zeros((n_epochs, n_channels, n_frband, n_time), dtype=np.float32)
    baseline_bands = np.zeros((n_epochs, n_channels, n_frband, n_time), dtype=np.float32)

    for i in range(len(band_indices)-1):
        power_bands[:, :, i, :] = np.mean(tmp_tfr[:, :, band_indices[i]:band_indices[i+1], :], axis=2)  
        tmp_baselined = tmp_tfr[:, :, band_indices[i]:band_indices[i+1], :] - tmp_baseline[:, :, band_indices[i]:band_indices[i+1], :]
        baseline_bands[:, :, i, :] = np.mean(tmp_baselined, axis=2)
    complete_tfr_band[subject] = power_bands.astype(np.float16)
    complete_baseline_band[subject] = baseline_bands.astype(np.float16)

del tmp_tfr, power_bands, baseline_bands, tmp_baselined
gc.collect()



In [ ]:
full_roc = {var: np.zeros((len(subjects), n_time//10 + 1)) for var in ["fb", "choice", "stim", "is_partial", "is_stimstable", "who_stable", "is_random", "good_strat", "hmm_switch", "firstswitch", "goodswitch"]}
full_roc_test = {var: np.zeros((len(subjects), n_time)) for var in ["fb"]}


In [ ]:

from sklearn.frozen import FrozenEstimator

n_time = complete_tfr_band[subject].shape[-1]
var = "is_stimstable"
# for subject in subjects:
subject = subjects[7]
n_epoch = complete_tfr[subject].shape[0]
for t in range(0, n_time, 10):
# t = 384
    X = complete_tfr_band[subject][:,:,:,t].reshape(n_epoch, -1)
    # for var in ["fb", "hmm_switch"]:
    y = complete_beh[complete_beh["subject"] == subject][var].values
    # _ = pipeline.fit(X, y)
    # y_pred = pipeline.predict(X)
    # y_proba = pipeline.predict_proba(X)
    # # roc = roc_auc_score(y, y_proba, multi_class="ovr")
    # roc = roc_auc_score(y, y_proba[:, 1])
    # f1 = f1_score(y, y_pred, average="weighted")
    # full_roc_test[var][subjects.index(subject), t] = roc
    # tmp_roc = []
    # tmp_f1 = []
    for train_idx, test_idx in cv.split(X, y):
        y_train, y_test = y[train_idx], y[test_idx]
        X_train = X[train_idx]#imputer.fit_transform(X[train_idx])
        X_test = X[test_idx] #imputer.fit_transform(X[test_idx])
        # _ = pipeline.fit(X_train, y_train)
        pipeline.fit(X_train, y_train)
        frozen_clf = FrozenEstimator(pipeline)
        # frozen_clf.fit(X_train, y_train)
        calib = CalibratedClassifierCV(estimator=frozen_clf, method="sigmoid", n_jobs=-1)
        calib.fit(X_train, y_train)
        y_pred = calib.predict(X_test)
        y_proba = calib.predict_proba(X_test)
        # roc = roc_auc_score(y_test, y_proba[:, 1])
        roc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
        f1 = f1_score(y_test, y_pred, average="weighted")
    #     y_pred = pipeline.predict(X_test)
    #     y_proba = pipeline.predict_proba(X_test)
    #     # y_score = y_proba[:, 1]  # proba classe positive
    #     roc = roc_auc_score(y_test, y_proba, multi_class="ovr")
    #     f1 = f1_score(y_test, y_pred, average="weighted")
    #     # roc = roc_auc_score(y_test, y_score)
    #     tmp_roc.append(roc)
    #     tmp_f1.append(f1)
    # roc = np.mean(tmp_roc)
    # f1 = np.mean(tmp_f1)
    print(f"Time : {t} - ROC AUC = {roc:.3f}, F1 Score = {f1:.3f}")
    full_roc[var][subjects.index(subject), t//10] = roc




In [ ]:
plt.plot(full_roc[var][7, :], alpha=0.8, color='blue', lw = 0.5)
plt.ylim(0.4, 0.7)
plt.axhline(0.5, color='black', linestyle='--', lw = 0.5)
plt.axvline((sr_decimated * 1.5)//10, color='red', linestyle='--', lw = 1)

In [ ]:
df_anat = group_selection.anatomy_data.copy()

In [ ]:
# idx_chan_motor = df_anat[(df_anat['region'] == 'Somatosensory') | (df_anat['region'] == 'Motor')].index
idx_chan_motor = df_anat[(df_anat['area_order'] < 5)].index
# chan_motor
chan_motor = df_anat.loc[idx_chan_motor]
chan_motor
# df_anat["region"].unique()
# # chan_motor["subject"].unique()
# subjects = chan_motor["subject"].unique()
# # n_epoch = complete_tfr[subject].shape[0]
# subject = subjects[0]

# channels_motor = chan_motor[chan_motor["subject"] == subject]["chan_idx"].values
# X = complete_tfr[subject][:,:,channels_motor][:,:,:,t].reshape(n_epoch, -1)


In [ ]:
chan_motor.groupby('subject').size()

In [ ]:
n_time = complete_tfr[subject].shape[-1]
var = "choice"
subjects = chan_motor["subject"].unique()
full_roc = {var: np.zeros((len(subjects), n_time//10 + 1)) for var in ["choice"]}
# for i, subject in enumerate(subjects):
subject = 28 #subjects[3]
i = len(subjects)-1
print(f"Subject {subject}")
n_epoch = complete_tfr[subject].shape[0]
channels_motor = chan_motor[chan_motor["subject"] == subject]["chan_idx"].values
for t in range(0, n_time, 10):
    X = complete_tfr[subject][:, channels_motor, :, t].reshape(n_epoch, -1)
    y = complete_beh[complete_beh["subject"] == subject][var].values
    tmp_roc, tmp_f1 = [], []
    for train_idx, test_idx in cv.split(X, y):
        y_train, y_test = y[train_idx], y[test_idx]
        X_train = imputer.fit_transform(X[train_idx])
        X_test = imputer.transform(X[test_idx])
        _ = pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)
        roc = roc_auc_score(y_test, y_proba, multi_class="ovr")
        f1 = f1_score(y_test, y_pred, average="weighted")
        tmp_roc.append(roc)
        tmp_f1.append(f1)
    roc = np.mean(tmp_roc)
    f1 = np.mean(tmp_f1)
    print(f"Time : {t} - ROC AUC = {roc:.3f}, F1 Score = {f1:.3f}")
    full_roc[var][i, t//10] = roc


In [ ]:
plt.plot(full_roc[var][-1, :].T, alpha=0.8, color='blue', lw = 0.5)
plt.show()

In [ ]:
avr_tc = np.mean(full_roc[var][:-1, :], axis=0)
sem_tc = np.std(full_roc[var][:-1, :], axis=0)/np.sqrt(len(subjects))
plt.plot(full_roc[var][:-1, :].T, alpha=0.8, color='blue', lw = 0.5)
plt.fill_between(range(len(avr_tc)), avr_tc - sem_tc, avr_tc + sem_tc, color='blue', alpha=0.3)
plt.plot(avr_tc, color='blue')
plt.ylim(0.4, 0.7)
plt.axhline(0.5, color='red', linestyle='--')
plt.show()

In [ ]:


subjects

In [ ]:
class NaivelyCalibratedLinearSVC(LinearSVC):
    """LinearSVC with `predict_proba` method that naively scales
    `decision_function` output."""

    def fit(self, X, y):
        super().fit(X, y)
        df = self.decision_function(X)
        self.df_min_ = df.min()
        self.df_max_ = df.max()

    def predict_proba(self, X):
        df = self.decision_function(X)
        if df.ndim == 1:
            # binaire
            p1 = 1.0 / (1.0 + np.exp(-df))
            return np.c_[1 - p1, p1]
        # multiclasses (ovr)
        df = df - df.max(axis=1, keepdims=True)  # stabilité num.
        exps = np.exp(df)
        probs = exps / exps.sum(axis=1, keepdims=True)
        return probs


In [ ]:
# svc = NaivelyCalibratedLinearSVC(max_iter=10000)
# pipeline = make_pipeline(StandardScaler(), svc)
clf = CalibratedClassifierCV(LinearSVC(max_iter=10000), cv=3, n_jobs=-1)
pipeline = make_pipeline(StandardScaler(), clf)

In [ ]:
complete_chan_roc = {subject: np.zeros((len(df_anat[df_anat["subject"] == subject]), n_time//10 + 1)) for subject in subjects}

# test_decode = np.zeros((len(df_anat[df_anat["subject"] == 2]), n_time//10 + 1))


In [ ]:
n_time = complete_tfr_band[subject].shape[-1]
var = "is_random"
subjects = df_anat["subject"].unique()
subject = subjects[-2]
# i = 0
print(f"Subject {subject}")
n_epoch = complete_tfr_band[subject].shape[0]
y = complete_beh[complete_beh["subject"] == subject][var].values
for chan in range(len(df_anat[df_anat["subject"] == subject])):
    X_ = complete_tfr_band[subject][:, chan, :, :]
    for t in range(0, n_time, 10):
        X = X_[:, :, t].reshape(n_epoch, -1)
        tmp_roc, tmp_f1 = [], []
        for train_idx, test_idx in cv.split(X, y):
            y_train, y_test = y[train_idx], y[test_idx]
            X_train = imputer.fit_transform(X[train_idx])
            X_test = imputer.transform(X[test_idx])
            _ = pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            y_proba = pipeline.predict_proba(X_test)

            n_classes = np.unique(y_train).size
            if n_classes == 2:
                roc = roc_auc_score(y_test, y_proba[:, 1])
            else:
                roc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
            f1 = f1_score(y_test, y_pred, average="weighted")
            tmp_roc.append(roc)
            tmp_f1.append(f1)
        roc = np.mean(tmp_roc)
        f1 = np.mean(tmp_f1)
        print(f"Time : {t}, Channel : {chan} | ROC AUC = {roc:.3f}, F1 Score = {f1:.3f}")
        complete_chan_roc[subject][chan, t//10] = roc

In [ ]:
subject = subjects[-2]
var = "is_random"
vmax = np.max(complete_chan_roc[subject])
# vmax = np.max(full_roc[var])

vmin = 0.50

plt.imshow(complete_chan_roc[subject], vmin=vmin, vmax=vmax, aspect='auto', cmap='jet')
# plt.imshow(full_roc[var], vmin=vmin, vmax=vmax, aspect='auto', cmap='jet')

plt.axvline((sr_decimated*2)//10, color='red', linestyle='--')
plt.yticks(
    ticks=np.arange(len(df_anat[df_anat["subject"] == subject])),
    labels=df_anat[df_anat["subject"] == subject]["region"].values
)
plt.colorbar()
plt.show()

In [ ]:
# avr_tc = np.mean(complete_chan_roc[subject], axis=0)
# sem_tc = np.std(complete_chan_roc[subject], axis=0)/np.sqrt(complete_chan_roc[subject].shape[0])
# plt.plot(complete_chan_roc[subject].T, lw = 0.5)
# plt.fill_between(range(len(avr_tc)), avr_tc - sem_tc, avr_tc + sem_tc, color='blue', alpha=0.3)
# plt.plot(avr_tc, color='blue')
# plt.ylim(0.48, 0.8)
# plt.axhline(0.5, color='red', linestyle='--')
# plt.show()

In [ ]:
coords = df_anat.loc[np.isin(df_anat['subject'], subject),
                        ["x", "y", "z", "region", "subject", "chan_idx"]].copy()
coords = coords.reset_index(drop=True)

ijk = np.round(apply_affine(np.linalg.inv(mni_tmp.affine),
                            coords[["x", "y", "z"]].to_numpy())).astype(int)
coords["i"], coords["j"], coords["k"] = ijk[:, 0], ijk[:, 1], ijk[:, 2]


In [ ]:
# i_mid = mni_tmp.shape[1]//2
slices_idx = np.arange(mni_tmp.shape[1]//2+10, mni_tmp.shape[1]-20, (mni_tmp.shape[1]//2-30)//9)
fig, axs = plt.subplots(3, 3, figsize=(15, 15))
axs = axs.ravel()

for i, ax in enumerate(axs):
    slice_idx = slices_idx[i]
    slice_img = mni_tmp.get_fdata()[:, slice_idx, :].T
    ax.imshow(slice_img, cmap='gray', origin='lower')
    csub = coords[coords["subject"] == subject]
    for roi, group in csub.groupby('region'):
        j_slice = np.abs(group["j"] - slice_idx) < 5
        color = area_colors[roi]
        idx = group["chan_idx"].values
        ax.scatter(group.loc[j_slice, "i"], group.loc[j_slice, "k"],
                    c=color, s = (((np.max(complete_chan_roc[subject][idx, :]))-0.5)*40)**2)
    ax.set_title(f'Slice j={slice_idx}')
    # ax.legend()

In [ ]:
n_time = complete_tfr_band[subject].shape[-1]
var = "is_stimstable"
subjects = chan_motor["subject"].unique()
# for i, subject in enumerate(subjects):
subject = subjects[6]
i = 6
print(f"Subject {subject}")
n_epoch = complete_tfr_band[subject].shape[0]
# channels_motor = chan_motor[chan_motor["subject"] == subject]["chan_idx"].values
for t in range(0, n_time, 10):
    X = complete_tfr_band[subject][:, :, :, t].reshape(n_epoch, -1)
    y = complete_beh[complete_beh["subject"] == subject][var].values
    tmp_roc, tmp_f1 = [], []
    for train_idx, test_idx in cv.split(X, y):
        y_train, y_test = y[train_idx], y[test_idx]
        X_train = imputer.fit_transform(X[train_idx])
        X_test = imputer.transform(X[test_idx])
        _ = pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)

        n_classes = np.unique(y_train).size
        if n_classes == 2:
            roc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            roc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
        f1 = f1_score(y_test, y_pred, average="weighted")
        tmp_roc.append(roc)
        tmp_f1.append(f1)
    roc = np.mean(tmp_roc)
    f1 = np.mean(tmp_f1)
    print(f"Time : {t} - ROC AUC = {roc:.3f}, F1 Score = {f1:.3f}")
    full_roc[var][i, t//10] = roc

In [ ]:
var = "is_stimstable"
plt.plot(full_roc[var][[0, 6], :].T)
plt.ylim(0.45, 0.65)
plt.axvline((sr_decimated*2)//10, color='red', linestyle='--')
plt.axhline(0.5, color='black', linestyle='--', lw=0.5)
plt.legend()
plt.show()


# full_roc[var][6, :]

In [ ]:
# plt.plot(full_roc[var][[0, 2, 8], :].T, alpha=0.8, color='blue', lw = 0.5)

avr_tc = np.mean(full_roc[var][[0, 2, 4, 8], :], axis=0)
sem_tc = np.std(full_roc[var][[0, 2, 4, 8], :], axis=0)/np.sqrt(len(subjects))
plt.plot(full_roc[var][[0, 2, 4, 8], :].T, alpha=0.8, lw = 1, label = subjects[[0, 2, 4, 8]])
plt.fill_between(range(len(avr_tc)), avr_tc - sem_tc, avr_tc + sem_tc, color='k', alpha=0.1)
plt.plot(avr_tc, color='k', lw = 0.5)
plt.ylim(0.45, 0.65)
plt.axvline((sr_decimated*2)//10, color='red', linestyle='--')
plt.axhline(0.5, color='black', linestyle='--', lw=0.5)
plt.legend()
plt.show()



In [ ]:
import nilearn.datasets
from nibabel.affines import apply_affine

mni_tmp = nilearn.datasets.load_mni152_template()
# np.dot(mni_tmp.affine, coords.loc[0, ['y', 'z']].values)

# plt.imshow(mni_tmp.get_fdata()[mni_tmp.shape[0]//2, :, :].T, cmap='gray', origin='lower')
# plt.show()

In [ ]:
coords = chan_motor.loc[np.isin(chan_motor['subject'], subjects[[0, 2, 4, 8]]),
                        ["x", "y", "z", "region", "subject"]].copy()
coords = coords.reset_index(drop=True)

ijk = np.round(apply_affine(np.linalg.inv(mni_tmp.affine),
                            coords[["x", "y", "z"]].to_numpy())).astype(int)
coords["i"], coords["j"], coords["k"] = ijk[:, 0], ijk[:, 1], ijk[:, 2]


In [ ]:
[area_colors[area] for area in csub['region']]

In [ ]:
i_mid = mni_tmp.shape[0] // 2
slice_img = mni_tmp.get_fdata()[i_mid, :, :].T

fig, axs = plt.subplots(2, 2, figsize=(8, 6), sharex=True, sharey=True)
axs = axs.flatten()
for ax, subj in zip(axs, subjects[[0, 2, 4, 8]]):
    ax.imshow(slice_img, cmap='gray', origin='lower')
    csub = coords[coords["subject"] == subj]
    for roi, group in csub.groupby('region'):
        color = area_colors[roi]
        ax.scatter(group.loc[:, "j"], group.loc[:, "k"],
                   c=color, s=20)
    ax.set_title(f"Subject {subj}")
    ax.set_xlabel("Y (vox)"); ax.set_ylabel("Z (vox)")
    ax.axhline(0, color='grey', lw=0.5, ls='--')
    ax.axvline(0, color='grey', lw=0.5, ls='--')
    

fig.tight_layout()
plt.show()

In [ ]:
coords

In [ ]:
avr_tc = np.mean(full_roc[var][:-1, :], axis=0)
sem_tc = np.std(full_roc[var][:-1, :], axis=0)/np.sqrt(len(subjects))
plt.plot(full_roc[var][:-1, :].T, alpha=0.8, color='blue', lw = 0.5)
plt.fill_between(range(len(avr_tc)), avr_tc - sem_tc, avr_tc + sem_tc, color='blue', alpha=0.3)
plt.plot(avr_tc, color='blue')
plt.ylim(0.4, 1.0)
plt.axvline(38.4, color='red', linestyle='--')
plt.axhline(0.5, color='black', linestyle='--', lw=0.5)
plt.show()

In [ ]:
n_time = complete_tfr_band[subject].shape[-1]
# var = "choice"
subjects = df_anat["subject"].unique()
for i, subject in enumerate(subjects):
    print(f"Subject {subject}")
    n_epoch = complete_tfr_band[subject].shape[0]
    # channels_motor = chan_motor[chan_motor["subject"] == subject]["chan_idx"].values
    for t in range(0, n_time, 10):
        X = complete_tfr_band[subject][:, :, :, t].reshape(n_epoch, -1)
        for var in ["is_partial", "is_random", "who_stable", "firstswitch", "goodswitch", "good_strat"]:
            y = complete_beh[complete_beh["subject"] == subject][var].values
            if len(np.unique(y)) > 1:
                if sum(y) > 5:
                    tmp_roc, tmp_f1 = [], []
                    for train_idx, test_idx in cv.split(X, y):
                        y_train, y_test = y[train_idx], y[test_idx]
                        X_train = imputer.fit_transform(X[train_idx])
                        X_test = imputer.transform(X[test_idx])
                        _ = pipeline.fit(X_train, y_train)
                        y_pred = pipeline.predict(X_test)
                        y_proba = pipeline.predict_proba(X_test)

                        n_classes = np.unique(y_train).size
                        if n_classes == 2:
                            roc = roc_auc_score(y_test, y_proba[:, 1])
                        else:
                            roc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
                        f1 = f1_score(y_test, y_pred, average="weighted")
                        tmp_roc.append(roc)
                        tmp_f1.append(f1)
                else :
                    tmp_roc, tmp_f1 = [0], [0]
            else:
                tmp_roc, tmp_f1 = [0], [0]
            roc = np.mean(tmp_roc)
            f1 = np.mean(tmp_f1)
            print(f"Time : {t} - ROC AUC = {roc:.3f}, F1 Score = {f1:.3f}")
            full_roc[var][i, t//10] = roc

In [ ]:
# plt.plot(full_roc[var][0, :], alpha=0.8, color='blue', lw = 0.5)
# plt.show()
# subjects[1:]
# full_roc[var][1:]
complete_beh.columns

In [ ]:
avr_tc = np.mean(full_roc[var], axis=0)
sem_tc = np.std(full_roc[var], axis=0)/np.sqrt(len(subjects))
plt.plot(full_roc[var].T, alpha=0.8, color='blue', lw = 0.5)
plt.fill_between(range(len(avr_tc)), avr_tc - sem_tc, avr_tc + sem_tc, color='blue', alpha=0.3)
plt.plot(avr_tc, color='blue')
plt.ylim(0.4, 1.0)
plt.axvline(38.4, color='red', linestyle='--')
plt.axhline(0.5, color='black', linestyle='--', lw=0.5)
plt.title(var)
plt.show()

In [ ]:
# complete_tfr_band.keys()
# complete_beh.loc[complete_beh["who_stable"].isna(), "who_stable"] = 0




In [ ]:
# from sklearn.svm import SVC

var = "fb"
var
# subjects = chan_motor["subject"].unique()
# subject = subjects[0]
# n_time = complete_tfr_band[subject].shape[-1]
# full_roc = {var: np.zeros((len(subjects), n_time//10 + 1)) for var in ["fb"]}
# svc = NaivelyCalibratedLinearSVC()
# pipeline = make_pipeline(StandardScaler(), svc)
# subjects = chan_motor["subject"].unique()
# subjects
# for i, subject in enumerate(subjects):
# # subject = 28 #subjects[3]
# i = len(subjects)-1
    print(f"Subject {subject}")
    n_epoch = complete_tfr_band[subject].shape[0]
    channels_motor = chan_motor[chan_motor["subject"] == subject]["chan_idx"].values
    for t in range(0, n_time, 10):
    # t = 384
        X = complete_tfr_band[subject][:, channels_motor, :, t].reshape(n_epoch, -1)
        y = complete_beh[complete_beh["subject"] == subject][var].values
        tmp_roc, tmp_f1 = [], []
        for train_idx, test_idx in cv.split(X, y):
            y_train, y_test = y[train_idx], y[test_idx]
            X_train = imputer.fit_transform(X[train_idx])
            X_test = imputer.transform(X[test_idx])
            _ = pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            y_proba = pipeline.predict_proba(X_test)

            n_classes = np.unique(y_train).size
            if n_classes == 2:
                roc = roc_auc_score(y_test, y_proba[:, 1])
            else:
                roc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
            f1 = f1_score(y_test, y_pred, average="weighted")
            tmp_roc.append(roc)
            tmp_f1.append(f1)
#         roc = np.mean(tmp_roc)
#         f1 = np.mean(tmp_f1)
#         print(f"Time : {t} - ROC AUC = {roc:.3f}, F1 Score = {f1:.3f}")
#         full_roc[var][i, t//10] = roc


In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(15, 12))
axs = axs.flatten()
for i, (k, v) in enumerate(full_roc.items()):
    mean_roc = np.mean(v, axis=1)
    # print(np.where(mean_roc == 0.0))
    non_zero_idx = np.where(mean_roc != 0.0)[0]
    avr_tc = np.mean(full_roc[k][non_zero_idx], axis=0)
    sem_tc = np.std(full_roc[k][non_zero_idx], axis=0)/np.sqrt(len(subjects))
    axs[i].plot(full_roc[k].T, alpha=0.8, color='blue', lw = 0.5)
    axs[i].fill_between(range(len(avr_tc)), avr_tc - sem_tc, avr_tc + sem_tc, color='blue', alpha=0.3)
    axs[i].plot(avr_tc, color='blue')
    axs[i].set_ylim(0.4, 0.7)
    axs[i].axvline((sr_decimated*2)//10, color='red', linestyle='--')
    axs[i].axhline(0.5, color='black', linestyle='--', lw=0.5)
    axs[i].set_title(k)
plt.tight_layout()
plt.show()

In [ ]:
var = "good_strat"
# avr_tc = np.mean(full_roc[var], axis=0)
# sem_tc = np.std(full_roc[var], axis=0)/np.sqrt(len(subjects))
fig, axs = plt.subplots(4,4, sharex=True, sharey=True)
axs = axs.flatten()
for i, subject in enumerate(subjects) :
    axs[i].plot(full_roc[var][i, :], alpha=1, lw = 1, label = subject)
    # plt.fill_between(range(len(avr_tc)), avr_tc - sem_tc, avr_tc + sem_tc, color='blue', alpha=0.3)
    # plt.plot(avr_tc, color='blue')
    axs[i].set_ylim(0.45, 0.75)
    axs[i].axvline((sr_decimated*2)//10, color='red', linestyle='--')
    axs[i].axhline(0.5, color='black', linestyle='--', lw=0.5)
    axs[i].set_title(subject)
plt.show()

In [ ]:
n_roi = df_anat['region'].nunique()
nrow = np.sqrt(n_roi).astype(int)
ncol = (n_roi // nrow) + 1
fig, axs = plt.subplots(nrow, ncol, figsize=(15, 10), sharex=True, sharey=True)
axs = axs.flatten()
n_time = complete_tfr_band[subject].shape[-1]
for i, (roi, group) in enumerate(df_anat.groupby('region')):
    tmp_switch = np.zeros((n_frband, n_time))
    tmp_prev_switch = np.zeros((n_frband, n_time))
    sub_counter = 0
    for subject in subjects:
        beh = complete_beh[complete_beh["subject"] == subject].reset_index()
        anat = group[group["subject"] == subject]
        if len(anat) > 0:
            chan_idx = anat['chan_idx'].values
            power = complete_tfr_band[subject][:, chan_idx, :, :]
            switch_idx = beh[(beh['is_stimstable'] == 1) & (beh['firstsw_pres'] == -1)].index
            # prev_switch_idx = []
            # for j in range(1, 4) :
            #     prev_switch_idx.append(switch_idx - j)
            #     prev_switch_idx.append(switch_idx + j)
            prev_switch_idx = beh[(beh['is_stimstable'] == 1) & (beh['firstsw_pres'] == 1)].index
            if (len(prev_switch_idx) != 0) & (len(switch_idx) != 0):
                # prev_switch_idx = np.concatenate(prev_switch_idx)
                # prev_switch_idx = prev_switch_idx[prev_switch_idx >= 0]
                switch_baseline = complete_baseline_band[subject][switch_idx, :, :, :][:, chan_idx, :, :]
                prev_switch_baseline = complete_baseline_band[subject][prev_switch_idx, :, :, :][:, chan_idx, :, :]
                # print(f"Subject {subject} - ROI {roi} - Switches : {len(switch_idx)}")
                switch_power = power[switch_idx, :, :, :] - switch_baseline
                prev_switch_power = power[prev_switch_idx, :, :, :] - prev_switch_baseline
                tmp_switch += np.mean(switch_power, axis=(0, 1))
                tmp_prev_switch += np.mean(prev_switch_power, axis=(0, 1))
                sub_counter += 1
    mean_switch = tmp_switch / sub_counter
    mean_prev_switch = tmp_prev_switch / sub_counter
    diff = mean_switch - mean_prev_switch
    # im = axs[i].imshow(diff, aspect='auto', origin='lower', cmap='jet', vmin=-np.max(np.abs(diff)), vmax=np.max(np.abs(diff)))
    axs[i].plot(mean_switch[[1, 2], :].T, label='Switch', color='blue')
    axs[i].plot(mean_prev_switch[[1, 2], :].T, label='Prev Switch', color='orange')
    axs[i].set_title(roi)
    axs[i].axvline((sr_decimated*2), color='black', linestyle='--')
    # axs[i].set_yticks(np.arange(n_frband), list(FREQUENCY_BANDS.keys()))
    # plt.colorbar(im, ax=axs[i])
axs[-1].plot([], [], label='Switch', color='blue')
axs[-1].plot([], [], label='Prev Switch', color='orange')
axs[-1].legend()
axs[-1].axis('off')
axs[-ncol].set_ylabel('Power Change (a.u.)')
axs[-ncol].set_xlabel('Time (ms)')
plt.suptitle(f'Frequency Bands: {list(FREQUENCY_BANDS.keys())[1]} and {list(FREQUENCY_BANDS.keys())[2]}')
plt.tight_layout()

In [ ]:

fig, axs = plt.subplots(nrow, ncol, figsize=(15, 10), sharex=True, sharey=True)
axs = axs.flatten()
n_time = complete_tfr_band[subject].shape[-1]
for i, (roi, group) in enumerate(df_anat.groupby('region')):
    tmp_switch = np.zeros((n_frband, n_time))
    tmp_prev_switch = np.zeros((n_frband, n_time))
    sub_counter = 0
    for subject in subjects:
        beh = complete_beh[complete_beh["subject"] == subject].reset_index()
        anat = group[group["subject"] == subject]
        if len(anat) > 0:
            chan_idx = anat['chan_idx'].values
            power = complete_tfr_band[subject][:, chan_idx, :, :]
            switch_idx = beh[(beh['is_stimstable'] == 1) & (beh['firstsw_pres'] == -1)].index
            # prev_switch_idx = []
            # for j in range(1, 4) :
            #     prev_switch_idx.append(switch_idx - j)
            #     prev_switch_idx.append(switch_idx + j)
            prev_switch_idx = beh[(beh['is_stimstable'] == 1) & (beh['firstsw_pres'] == 1)].index
            if (len(prev_switch_idx) != 0) & (len(switch_idx) != 0):
                # prev_switch_idx = np.concatenate(prev_switch_idx)
                # prev_switch_idx = prev_switch_idx[prev_switch_idx >= 0]
                switch_baseline = complete_baseline_band[subject][switch_idx, :, :, :][:, chan_idx, :, :]
                prev_switch_baseline = complete_baseline_band[subject][prev_switch_idx, :, :, :][:, chan_idx, :, :]
                # print(f"Subject {subject} - ROI {roi} - Switches : {len(switch_idx)}")
                switch_power = power[switch_idx, :, :, :] - switch_baseline
                prev_switch_power = power[prev_switch_idx, :, :, :] - prev_switch_baseline
                tmp_switch += np.mean(switch_power, axis=(0, 1))
                tmp_prev_switch += np.mean(prev_switch_power, axis=(0, 1))
                sub_counter += 1
    mean_switch = tmp_switch / sub_counter
    mean_prev_switch = tmp_prev_switch / sub_counter
    diff = mean_switch - mean_prev_switch
    # im = axs[i].imshow(diff, aspect='auto', origin='lower', cmap='jet', vmin=-np.max(np.abs(diff)), vmax=np.max(np.abs(diff)))
    axs[i].plot(mean_switch[[3, 4], :].T, label='Switch', color='blue')
    axs[i].plot(mean_prev_switch[[3, 4], :].T, label='Prev Switch', color='orange')
    axs[i].set_title(roi)
    axs[i].axvline((sr_decimated*2), color='black', linestyle='--')
    # axs[i].set_yticks(np.arange(n_frband), list(FREQUENCY_BANDS.keys()))
    # plt.colorbar(im, ax=axs[i])
axs[-1].plot([], [], label='Switch', color='blue')
axs[-1].plot([], [], label='Prev Switch', color='orange')
axs[-1].legend()
axs[-1].axis('off')
axs[-ncol].set_ylabel('Power Change (a.u.)')
axs[-ncol].set_xlabel('Time (ms)')
plt.suptitle(f'Frequency Bands: {list(FREQUENCY_BANDS.keys())[3]} and {list(FREQUENCY_BANDS.keys())[4]}')
plt.tight_layout()

In [ ]:

fig, axs = plt.subplots(nrow, ncol, figsize=(15, 10), sharex=True, sharey=True)
axs = axs.flatten()
n_time = complete_tfr_band[subject].shape[-1]
for i, (roi, group) in enumerate(df_anat.groupby('region')):
    tmp_switch = np.zeros((n_frband, n_time))
    tmp_prev_switch = np.zeros((n_frband, n_time))
    sub_counter = 0
    for subject in subjects:
        beh = complete_beh[complete_beh["subject"] == subject].reset_index()
        anat = group[group["subject"] == subject]
        if len(anat) > 0:
            chan_idx = anat['chan_idx'].values
            power = complete_tfr_band[subject][:, chan_idx, :, :]
            switch_idx = beh[(beh['is_stimstable'] == 1) & (beh['firstsw_pres'] == -1)].index
            # prev_switch_idx = []
            # for j in range(1, 4) :
            #     prev_switch_idx.append(switch_idx - j)
            #     prev_switch_idx.append(switch_idx + j)
            prev_switch_idx = beh[(beh['is_stimstable'] == 1) & (beh['firstsw_pres'] == 1)].index
            if (len(prev_switch_idx) != 0) & (len(switch_idx) != 0):
                # prev_switch_idx = np.concatenate(prev_switch_idx)
                # prev_switch_idx = prev_switch_idx[prev_switch_idx >= 0]
                switch_baseline = complete_baseline_band[subject][switch_idx, :, :, :][:, chan_idx, :, :]
                prev_switch_baseline = complete_baseline_band[subject][prev_switch_idx, :, :, :][:, chan_idx, :, :]
                # print(f"Subject {subject} - ROI {roi} - Switches : {len(switch_idx)}")
                switch_power = power[switch_idx, :, :, :] - switch_baseline
                prev_switch_power = power[prev_switch_idx, :, :, :] - prev_switch_baseline
                tmp_switch += np.mean(switch_power, axis=(0, 1))
                tmp_prev_switch += np.mean(prev_switch_power, axis=(0, 1))
                sub_counter += 1
    mean_switch = tmp_switch / sub_counter
    mean_prev_switch = tmp_prev_switch / sub_counter
    diff = mean_switch - mean_prev_switch
    # im = axs[i].imshow(diff, aspect='auto', origin='lower', cmap='jet', vmin=-np.max(np.abs(diff)), vmax=np.max(np.abs(diff)))
    axs[i].plot(mean_switch[[5, 6], :].T, label='Switch', color='blue')
    axs[i].plot(mean_prev_switch[[5, 6], :].T, label='Prev Switch', color='orange')
    axs[i].set_title(roi)
    axs[i].axvline((sr_decimated*2), color='black', linestyle='--')
    # axs[i].set_yticks(np.arange(n_frband), list(FREQUENCY_BANDS.keys()))
    # plt.colorbar(im, ax=axs[i])
axs[-1].plot([], [], label='Switch', color='blue')
axs[-1].plot([], [], label='Prev Switch', color='orange')
axs[-1].legend()
axs[-1].axis('off')
axs[-ncol].set_ylabel('Power Change (a.u.)')
axs[-ncol].set_xlabel('Time (ms)')
plt.suptitle(f'Frequency Bands: {list(FREQUENCY_BANDS.keys())[5]} and {list(FREQUENCY_BANDS.keys())[6]}', fontsize=26)
plt.tight_layout()

In [ ]:

# fig, axs = plt.subplots(nrow, ncol, figsize=(15, 10), sharex=True, sharey=True)
# axs = axs.flatten()
# n_time = complete_tfr_band[subject].shape[-1]
# fig, axs = plt.subplots(1, 1, figsize=(12, 10))
# for i, (roi, group) in enumerate(df_anat.groupby('region')):
roi = "VLPFC_POST"
group = df_anat[df_anat['region'] == roi]
tmp_switch = []
tmp_prev_switch = []
sub_counter = 0
for subject in subjects:
    beh = complete_beh[complete_beh["subject"] == subject].reset_index()
    anat = group[group["subject"] == subject]
    if len(anat) > 0:
        chan_idx = anat['chan_idx'].values
        power = complete_tfr_band[subject][:, chan_idx, :, :]
        switch_idx = beh[(beh['is_stimstable'] == 1) & (beh['firstsw_pres'] == -1)].index
        # switch_idx = beh[(beh['prev_fb'] == 1)].index
        # prev_switch_idx = beh[(beh['prev_fb'] == 0)].index
        # prev_switch_idx = []
        # for j in range(1, 4) :
        #     prev_switch_idx.append(switch_idx - j)
        #     prev_switch_idx.append(switch_idx + j)
        prev_switch_idx = beh[(beh['is_stimstable'] == 1) & (beh['firstsw_pres'] == 1)].index
        if (len(prev_switch_idx) != 0) & (len(switch_idx) != 0):
            # prev_switch_idx = np.concatenate(prev_switch_idx)
            # prev_switch_idx = prev_switch_idx[prev_switch_idx >= 0]
            switch_baseline = complete_baseline_band[subject][switch_idx, :, :, :][:, chan_idx, :, :]
            prev_switch_baseline = complete_baseline_band[subject][prev_switch_idx, :, :, :][:, chan_idx, :, :]
            # print(f"Subject {subject} - ROI {roi} - Switches : {len(switch_idx)}")
            switch_power = power[switch_idx, :, :, :] - switch_baseline
            prev_switch_power = power[prev_switch_idx, :, :, :] - prev_switch_baseline
            tmp_switch.append(np.mean(switch_power, axis=(0)))
            tmp_prev_switch.append(np.mean(prev_switch_power, axis=(0)))
            sub_counter += 1

# mean_switch = np.array(tmp_switch)
# mean_prev_switch = np.array(tmp_prev_switch)
# diff = mean_switch - mean_prev_switch
fig, axs = plt.subplots(3, 1, sharex=True, sharey=True)
# axs[0].plot(mean_switch[:, 6, :].T, color='blue')
# axs[1].plot(mean_prev_switch[:, 6, :].T, color='orange')
# # axs[2].plot(diff[:, 6, :].T, color='green')
# for ax in axs:
#     ax.axvline((sr_decimated*2), color='black', linestyle='--')
band = 3

for i in range(len(tmp_switch)):
    tmp_switch[i] = tmp_switch[i][:, band, :]
    tmp_prev_switch[i] = tmp_prev_switch[i][:, band, :]
    diff = tmp_switch[i] - tmp_prev_switch[i]
    axs[0].plot(tmp_switch[i].T, color='blue', alpha=0.3, lw=0.5)
    axs[0].axvline((sr_decimated*2), color='black', linestyle='--')
    axs[1].plot(tmp_prev_switch[i].T, color='orange', alpha=0.3, lw=0.5)
    axs[1].axvline((sr_decimated*2), color='black', linestyle='--')
    axs[2].plot(diff.T, color='green', alpha=0.3, lw=0.5)
    axs[2].axvline((sr_decimated*2), color='black', linestyle='--')





# avr_tc = np.zeros((n_time,))
# for mat in tmp_switch:
#     axs[0].plot(mat[:, band, :].T, color='blue', alpha=0.3, lw =0.5)
#     axs[0].axvline((sr_decimated*2), color='black', linestyle='--')
#     avr_tc += np.mean(mat[:, band, :], axis=0)
# # avr_tc /= sub_counter
# axs[0].plot(avr_tc, color='blue', lw=2)

# avr_tc = np.zeros((n_time,))
# for mat in tmp_prev_switch:
#     axs[1].plot(mat[:, band, :].T, color='orange', alpha=0.3, lw=0.5)
#     axs[1].axvline((sr_decimated*2), color='black', linestyle='--')
#     avr_tc += np.mean(mat[:, band, :], axis=0)
# avr_tc #/= sub_counter
# axs[1].plot(avr_tc, color='orange', lw=2)
# axs[0].set_title('Correct')
# axs[1].set_title('Incorrect')
plt.suptitle(f'Frequency Band: {list(FREQUENCY_BANDS.keys())[1]} - ROI: {roi}', fontsize=16)
plt.xlabel('Time (ms)')
plt.ylabel('Power Change (a.u.)')
plt.tight_layout()  
# mean_switch = #tmp_switch / sub_counter
# mean_prev_switch = #tmp_prev_switch / sub_counter
#     # diff = mean_switch - mean_prev_switch
# # im = axs[i].imshow(diff, aspect='auto', origin='lower', cmap='jet', vmin=-np.max(np.abs(diff)), vmax=np.max(np.abs(diff)))
# axs.plot(mean_switch[[5, 6], :].T, label='Switch', color='blue')
# axs.plot(mean_prev_switch[[5, 6], :].T, label='Prev Switch', color='orange')
# axs.set_title(roi)
# axs.axvline((sr_decimated*2), color='black', linestyle='--')
#     # axs[i].set_yticks(np.arange(n_frband), list(FREQUENCY_BANDS.keys()))
#     # plt.colorbar(im, ax=axs[i])
# # axs[-1].plot([], [], label='Switch', color='blue')
# # axs[-1].plot([], [], label='Prev Switch', color='orange')
# # axs[-1].legend()
# # axs[-1].axis('off')
# # axs[-ncol].set_ylabel('Power Change (a.u.)')
# # axs[-ncol].set_xlabel('Time (ms)')
# plt.suptitle(f'Frequency Bands: {list(FREQUENCY_BANDS.keys())[5]} and {list(FREQUENCY_BANDS.keys())[6]}', fontsize=26)
# plt.tight_layout()